In [ ]:
# news_digest.py — AI News Digest CLI
import os
import json
import argparse
from datetime import date
from pathlib import Path
import requests
import anthropic
from dotenv import load_dotenv
load_dotenv() # Load API keys from .env
NEWS_API_KEY = os.getenv("NEWS_API_KEY")
AI_CLIENT = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CACHE_FILE = Path("cache.json")
# ─── News Fetching ─────────────────────────────────────────────────────────
def fetch_articles(topic: str, count: int = 5) -> list[dict]:
 """Fetch top news articles on a topic from NewsAPI."""
 url = "https://newsapi.org/v2/everything"
 params = {
 "q": topic,
 "language": "en",
 "sortBy": "publishedAt",
 "pageSize": count,
 "apiKey": NEWS_API_KEY,
 }
 response = requests.get(url, params=params)
 response.raise_for_status() # raises an exception if request failed
 articles = response.json().get("articles", [])
 print(f"Fetched {len(articles)} articles about '{topic}'")
 return articles
# ─── AI Summarisation ──────────────────────────────────────────────────────
def summarise_article(title: str, content: str) -> str:
 """Use Claude to write a 2-3 sentence summary of an article."""
 if not content or len(content) < 100:
     return "Full article content not available for summarisation."
 message = AI_CLIENT.messages.create(
 model="claude-haiku-4-5-20251001", # Fast, cheap model for summaries
 max_tokens=200,
 messages=[{
 "role": "user",
 "content": (
 f"Summarise this news article in exactly 2-3 clear sentences. "
 f"Be factual and concise. No fluff.\n\n"
 f"Title: {title}\n\nContent: {content[:2000]}")}])
 return message.content[0].text
# ─── Digest Generation ─────────────────────────────────────────────────────
def build_digest(topic: str, articles: list[dict]) -> str:
 """Build a Markdown-formatted digest from articles."""
 today = date.today().isoformat()
 lines = []
 lines.append(f"# AI News Digest — {topic.title()}")
 lines.append(f"*Generated on {today} | {len(articles)} articles*")
 lines.append("")
 for i, article in enumerate(articles, start=1):
     title = article.get("title", "Untitled")
     source = article.get("source", {}).get("name", "Unknown Source")
     url = article.get("url", "")
     content = article.get("content") or article.get("description") or ""
     pub_date = article.get("publishedAt", "")[:10] # just the date part
     print(f" Summarising article {i}/{len(articles)}: {title[:50]}...")
     summary = summarise_article(title, content)
     lines.append(f"## {i}. {title}")
     lines.append(f"*Source: {source} | Published: {pub_date}*")
     lines.append("")
     lines.append(f"**Summary:** {summary}")
     lines.append("")
     lines.append(f"[Read full article]({url})")
     lines.append("")
     lines.append("---")
     lines.append("")
 return "\n".join(lines)
def save_digest(content: str, topic: str) -> Path:
 """Save digest to a markdown file."""
 today = date.today().isoformat()
 filename = f"digest_{topic.replace(' ', '_')}_{today}.md"
 filepath = Path(filename)
 filepath.write_text(content, encoding="utf-8")
 return filepath
# ─── CLI Entry Point ────────────────────────────────────────────────────────
def main(topic: str = "artificial intelligence", count: int = 5):
    print(f"\n Fetching news about: {topic}")
    articles = fetch_articles(topic, count)
    print("\n Generating AI summaries...")
    digest = build_digest(topic, articles)
    filepath = save_digest(digest, topic)
    print(f"\n Digest saved to: {filepath}")
    print("\nFirst 300 characters of your digest:")
    print(digest[:300])

# Run the main function with default parameters
main()
